In [3]:
# ============================================================
# ABL1 — MD6 SELF-CORRECTNESS = YES-ONLY TRAINING ABLATION
#
# PARSER FIX:
#   Raw MathDial does not always label student turns as "Student:".
#   Named speakers are normalized to the Student role.
#
# OPTIONAL KAGGLE RAW DATASET:
#   If you upload the official raw files as a Kaggle dataset, a clean
#   dataset name is: mathdial-raw
#   Expected files:
#       /kaggle/input/mathdial-raw/train.jsonl
#       /kaggle/input/mathdial-raw/test.jsonl
#   The notebook scans /kaggle/input recursively, so this path is
#   discovered automatically. Uploading is optional if HF download works.
#
# Research question:
#   Does training the best supervised selector only on MathDial
#   dialogues that ended with self-correctness = "Yes" improve
#   general pedagogical-move prediction?
#
# IMPORTANT DESIGN:
#   Main model baseline = MD6 Epoch 2
#   Only the TRAINING DIALOGUE SUBSET changes.
#
#   ABL1 training:
#       custom supervised TRAIN split
#       -> retain only dialogues with raw MathDial
#          self-correctness == "Yes"
#
#   Primary evaluation:
#       SAME COMPLETE 1,850-example validation set used by MD6
#
#   Secondary diagnostic:
#       validation examples belonging to self-correctness="Yes"
#       dialogues only
#
# self-correctness is OFFLINE FILTERING METADATA ONLY.
# It is NEVER added to problem/history model inputs.
#
# MD6 training recipe retained:
#   Model: distilbert/distilroberta-base
#   Max length: 512
#   alpha: 0.75
#   LR: 2e-5
#   Effective global batch: 16 (2 GPUs x batch 8)
#   Epochs: 3
#   Weight decay: 0.01
#   Warmup steps: 280
#   Seed: 42
#
# NOTE:
#   Class weights are recalculated on the filtered ABL1 training
#   subset using the SAME MD6 rule: inverse_frequency ** 0.75.
#
#   Because the training subset is smaller, 3 epochs contain fewer
#   optimizer updates than MD6. This is intentionally left unchanged
#   so the hyperparameter configuration stays fixed.
#
# Raw MathDial metadata:
#   The notebook first looks for raw MathDial JSONL files in Kaggle
#   inputs. If they are not present, it downloads eth-nlped/mathdial
#   from Hugging Face. In that case Kaggle Internet must be ON.
#
# CUSTOM HELD-OUT TEST SET: NOT LOADED / NOT EVALUATED.
# ============================================================

import gc
import json
import random
import re
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F

from datasets import Dataset, load_dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "distilbert/distilroberta-base"
MAX_LENGTH = 512
WEIGHT_ALPHA = 0.75

TRAIN_PATH = Path(
    "/kaggle/input/datasets/malindus/mathdial-move-selection/train.jsonl"
)
VAL_PATH = Path(
    "/kaggle/input/datasets/malindus/mathdial-move-selection/validation.jsonl"
)

OUTPUT_DIR = Path("/kaggle/working/abl1_md6_yes_only")
RESULT_DIR = Path("/kaggle/working/abl1_results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["generic", "probing", "focus", "telling"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 78)
print("ABL1 CONFIGURATION")
print("=" * 78)
print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)
print("Weight alpha:", WEIGHT_ALPHA)
print("GPU count:", torch.cuda.device_count())

assert torch.cuda.is_available(), "GPU is required."
assert torch.cuda.device_count() == 2, (
    "ABL1 is defined for Kaggle 2xT4, matching MD6. "
    f"Detected {torch.cuda.device_count()} GPU(s)."
)

def load_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows_full = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)

assert len(train_rows_full) == 14905
assert len(val_rows) == 1850

required = {"example_id", "qid", "problem", "history", "target_move"}
for name, rows in [
    ("train", train_rows_full),
    ("validation", val_rows),
]:
    assert required.issubset(rows[0].keys()), f"{name} schema mismatch"
    bad = [r for r in rows if r["target_move"] not in LABEL2ID]
    assert not bad, f"{name}: invalid labels found"

print("Original train move examples:", len(train_rows_full))
print("Validation move examples:", len(val_rows))
print("Custom held-out test loaded: False")


ABL1 CONFIGURATION
Model: distilbert/distilroberta-base
Max length: 512
Weight alpha: 0.75
GPU count: 2
Original train move examples: 14905
Validation move examples: 1850
Custom held-out test loaded: False


In [4]:
# ============================================================
# RECOVER RAW self-correctness METADATA AND BUILD YES-ONLY TRAIN
# ============================================================
#
# Why this matching is safe:
# - The fixed custom split was qid-leakage-safe, so train and
#   validation qids are disjoint.
# - Raw MathDial has multiple dialogue scenarios per qid.
# - We reconstruct every labelled teacher-move example from each
#   raw dialogue and match it to the already-fixed processed rows
#   using:
#       qid + problem + observable history + target move
#
# Before filtering, we REQUIRE reconstructed raw move-example
# counters to match the processed train/validation counters exactly.
# If the parser/mapping is wrong, the notebook stops BEFORE training.
# ============================================================

def canonical_text(x):
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x).strip())

def canonical_qid(x):
    return canonical_text(x)

def canonical_role(x):
    return canonical_text(x).lower()

def example_signature(qid, problem, history, target_move):
    canonical_history = tuple(
        (
            canonical_role(turn["user"]),
            canonical_text(turn["text"]),
        )
        for turn in history
    )
    return (
        canonical_qid(qid),
        canonical_text(problem),
        canonical_history,
        canonical_text(target_move).lower(),
    )

def processed_signature(ex):
    return example_signature(
        ex["qid"],
        ex["problem"],
        ex["history"],
        ex["target_move"],
    )

# ------------------------------------------------------------
# Load raw MathDial.
# First prefer any raw JSONL already attached as Kaggle input.
# Otherwise use the official Hugging Face dataset.
# ------------------------------------------------------------

RAW_REQUIRED = {
    "qid",
    "question",
    "conversation",
    "self-correctness",
}

def inspect_jsonl_schema(path):
    try:
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    obj = json.loads(line)
                    return set(obj.keys())
    except Exception:
        return set()
    return set()

raw_jsonl_paths = []
for path in Path("/kaggle/input").glob("**/*.jsonl"):
    schema = inspect_jsonl_schema(path)
    if RAW_REQUIRED.issubset(schema):
        raw_jsonl_paths.append(path)

raw_dialogues = []

if raw_jsonl_paths:
    print("Using raw MathDial JSONL files from Kaggle input:")
    for path in raw_jsonl_paths:
        print("  ", path)
        raw_dialogues.extend(load_jsonl(path))
else:
    print(
        "Raw MathDial JSONL not found in Kaggle inputs.\n"
        "Loading official eth-nlped/mathdial from Hugging Face..."
    )
    raw_ds = load_dataset("eth-nlped/mathdial")
    for split_name in raw_ds.keys():
        raw_dialogues.extend(dict(row) for row in raw_ds[split_name])

print("Raw MathDial dialogues loaded:", len(raw_dialogues))
assert len(raw_dialogues) == 2861, (
    "Expected the complete official MathDial corpus (2,861 dialogues). "
    f"Found {len(raw_dialogues)}."
)

# ------------------------------------------------------------
# Parse a raw MathDial conversation into the same observable
# role/text history representation used by the selector.
#
# IMPORTANT RAW-DATA QUIRK:
# Student turns may be prefixed by "Student:" OR by the student's
# actual name. Only "Teacher:" is treated as the teacher; all other
# speaker prefixes are normalized to "Student".
# ------------------------------------------------------------

MOVE_PATTERN = re.compile(
    r"^\(\s*(generic|probing|focus|telling)\s*\)\s*(.*)$",
    flags=re.IGNORECASE | re.DOTALL,
)

def split_raw_segment(segment):
    """
    MathDial teacher turns are explicitly prefixed with "Teacher:".

    Student turns are NOT always prefixed with "Student:". Some raw
    dialogues use the student's name instead (for example "Steven:",
    "Mariana:", etc.). Therefore:
      - prefix == Teacher  -> Teacher
      - every other prefix -> Student

    We split only at the FIRST colon, matching the raw conversation
    format while normalizing the observable runtime role to Student.
    """
    if ":" not in segment:
        raise ValueError(
            "Could not parse MathDial conversation segment "
            "(no colon found):\n" + repr(segment[:500])
        )

    speaker_prefix, body = segment.split(":", 1)
    speaker_prefix = speaker_prefix.strip()
    body = body.strip()

    role = (
        "Teacher"
        if speaker_prefix.lower() == "teacher"
        else "Student"
    )

    return role, body


def raw_dialogue_move_signatures(row):
    history = []
    move_keys = []

    conversation = str(row.get("conversation", ""))

    for raw_segment in conversation.split("|EOM|"):
        segment = raw_segment.strip()
        if not segment:
            continue

        role, body = split_raw_segment(segment)

        if role == "Teacher":
            move_match = MOVE_PATTERN.match(body)

            if move_match is not None:
                move = move_match.group(1).lower()
                teacher_text = move_match.group(2).strip()

                # Target is the CURRENT teacher move. Runtime history
                # contains only turns that came before it.
                move_keys.append(
                    example_signature(
                        row["qid"],
                        row["question"],
                        history,
                        move,
                    )
                )
            else:
                # Keep an unlabeled teacher utterance in history if present,
                # because later observable history would still contain it.
                teacher_text = body

            history.append({
                "user": "Teacher",
                "text": teacher_text,
            })

        else:
            history.append({
                "user": "Student",
                "text": body,
            })

    return move_keys

def normalize_outcome(value):
    s = canonical_text(value)

    if s == "Yes":
        return "Yes"

    if s in {
        "Yes, but I had to reveal the answer",
        "Yes but reveal",
    }:
        return "Yes but reveal"

    if s == "No":
        return "No"

    return "missing"

# ------------------------------------------------------------
# Corpus-level parser sanity audit.
# Official MathDial contains 18,607 labelled teacher moves and
# 2 unlabeled teacher segments. This catches gross parser errors
# before we attempt the custom-split reconstruction.
# ------------------------------------------------------------

total_raw_labelled_moves = 0
total_raw_unlabelled_teacher_segments = 0

for row in raw_dialogues:
    total_raw_labelled_moves += len(
        raw_dialogue_move_signatures(row)
    )

    for raw_segment in str(row.get("conversation", "")).split("|EOM|"):
        segment = raw_segment.strip()
        if not segment:
            continue

        role, body = split_raw_segment(segment)

        if role == "Teacher" and MOVE_PATTERN.match(body) is None:
            total_raw_unlabelled_teacher_segments += 1

print("\nRaw parser corpus audit:")
print("Labelled teacher moves:", total_raw_labelled_moves)
print(
    "Unlabelled teacher segments:",
    total_raw_unlabelled_teacher_segments,
)

assert total_raw_labelled_moves == 18607
assert total_raw_unlabelled_teacher_segments == 2

print("Raw parser corpus audit: PASSED")


train_qids = {
    canonical_qid(r["qid"])
    for r in train_rows_full
}

val_qids = {
    canonical_qid(r["qid"])
    for r in val_rows
}

assert train_qids.isdisjoint(val_qids)

raw_train_dialogues = [
    row
    for row in raw_dialogues
    if canonical_qid(row["qid"]) in train_qids
]

raw_val_dialogues = [
    row
    for row in raw_dialogues
    if canonical_qid(row["qid"]) in val_qids
]

print("\nRecovered custom split dialogue counts:")
print("Train dialogues:", len(raw_train_dialogues))
print("Validation dialogues:", len(raw_val_dialogues))

assert len(raw_train_dialogues) == 2295
assert len(raw_val_dialogues) == 278

# ------------------------------------------------------------
# Critical exact-matching audit.
# Reconstruct ALL move examples from raw dialogues and require
# exact multiset equality with the fixed processed datasets.
# ------------------------------------------------------------

processed_train_counter = Counter(
    processed_signature(ex)
    for ex in train_rows_full
)

processed_val_counter = Counter(
    processed_signature(ex)
    for ex in val_rows
)

raw_train_counter = Counter()
for row in raw_train_dialogues:
    raw_train_counter.update(
        raw_dialogue_move_signatures(row)
    )

raw_val_counter = Counter()
for row in raw_val_dialogues:
    raw_val_counter.update(
        raw_dialogue_move_signatures(row)
    )

if raw_train_counter != processed_train_counter:
    missing = processed_train_counter - raw_train_counter
    extra = raw_train_counter - processed_train_counter
    raise RuntimeError(
        "Raw->processed TRAIN reconstruction mismatch. "
        f"Missing processed signatures={sum(missing.values())}; "
        f"extra raw signatures={sum(extra.values())}. "
        "Stop: do not run the ablation until mapping is exact."
    )

if raw_val_counter != processed_val_counter:
    missing = processed_val_counter - raw_val_counter
    extra = raw_val_counter - processed_val_counter
    raise RuntimeError(
        "Raw->processed VALIDATION reconstruction mismatch. "
        f"Missing processed signatures={sum(missing.values())}; "
        f"extra raw signatures={sum(extra.values())}. "
        "Stop: do not run the ablation until mapping is exact."
    )

print("\nRaw/processed reconstruction audit: PASSED")
print("Train reconstructed moves:", sum(raw_train_counter.values()))
print("Validation reconstructed moves:", sum(raw_val_counter.values()))

# ------------------------------------------------------------
# Verify the previously audited train outcome distribution.
# ------------------------------------------------------------

train_outcome_counts = Counter(
    normalize_outcome(row.get("self-correctness"))
    for row in raw_train_dialogues
)

print("\nCustom TRAIN self-correctness distribution:")
for outcome in ["Yes", "Yes but reveal", "No", "missing"]:
    print(
        f"{outcome:15s}: "
        f"{train_outcome_counts[outcome]:4d}"
    )

EXPECTED_TRAIN_OUTCOMES = {
    "Yes": 1695,
    "Yes but reveal": 330,
    "No": 260,
    "missing": 10,
}

assert dict(train_outcome_counts) == EXPECTED_TRAIN_OUTCOMES, (
    "Outcome audit does not match the established custom-train counts."
)

# ------------------------------------------------------------
# Build exact occurrence-aware YES-only move counters.
# ------------------------------------------------------------

yes_train_counter = Counter()
yes_train_dialogues = 0

for row in raw_train_dialogues:
    if normalize_outcome(row.get("self-correctness")) == "Yes":
        yes_train_dialogues += 1
        yes_train_counter.update(
            raw_dialogue_move_signatures(row)
        )

assert yes_train_dialogues == 1695

# Consume signature counts in processed-row order.
remaining_yes_train = yes_train_counter.copy()
train_rows = []

for ex in train_rows_full:
    key = processed_signature(ex)
    if remaining_yes_train[key] > 0:
        train_rows.append(ex)
        remaining_yes_train[key] -= 1

assert sum(remaining_yes_train.values()) == 0
assert len(train_rows) == sum(yes_train_counter.values())

print("\nABL1 filtered training set:")
print("Yes dialogues:", yes_train_dialogues)
print("Move examples:", len(train_rows))
print(
    "Retained move-example fraction:",
    f"{len(train_rows) / len(train_rows_full):.2%}",
)

# ------------------------------------------------------------
# Secondary YES-only validation diagnostic mask.
# PRIMARY evaluation remains the full 1,850-example validation set.
# ------------------------------------------------------------

yes_val_counter = Counter()
yes_val_dialogues = 0

for row in raw_val_dialogues:
    if normalize_outcome(row.get("self-correctness")) == "Yes":
        yes_val_dialogues += 1
        yes_val_counter.update(
            raw_dialogue_move_signatures(row)
        )

remaining_yes_val = yes_val_counter.copy()
val_yes_mask_list = []

for ex in val_rows:
    key = processed_signature(ex)

    is_yes = remaining_yes_val[key] > 0
    val_yes_mask_list.append(is_yes)

    if is_yes:
        remaining_yes_val[key] -= 1

assert sum(remaining_yes_val.values()) == 0

val_yes_mask = np.asarray(
    val_yes_mask_list,
    dtype=bool,
)

print("\nSecondary validation diagnostic:")
print("Yes validation dialogues:", yes_val_dialogues)
print("Yes validation move examples:", int(val_yes_mask.sum()))
print("Full validation examples (PRIMARY):", len(val_rows))

# Save a non-privileged copy of the filtered processed rows for audit.
# Notice self-correctness itself is NOT written into selector examples.
filtered_path = RESULT_DIR / "abl1_yes_only_train_examples.jsonl"

with filtered_path.open("w", encoding="utf-8") as f:
    for ex in train_rows:
        f.write(json.dumps(ex) + "\n")

print("Filtered training rows saved:", filtered_path)


Using raw MathDial JSONL files from Kaggle input:
   /kaggle/input/datasets/malindus/mathdial-raw/test.jsonl
   /kaggle/input/datasets/malindus/mathdial-raw/train.jsonl
Raw MathDial dialogues loaded: 2861

Raw parser corpus audit:
Labelled teacher moves: 18607
Unlabelled teacher segments: 2
Raw parser corpus audit: PASSED

Recovered custom split dialogue counts:
Train dialogues: 2295
Validation dialogues: 278

Raw/processed reconstruction audit: PASSED
Train reconstructed moves: 14905
Validation reconstructed moves: 1850

Custom TRAIN self-correctness distribution:
Yes            : 1695
Yes but reveal :  330
No             :  260
missing        :   10

ABL1 filtered training set:
Yes dialogues: 1695
Move examples: 9798
Retained move-example fraction: 65.74%

Secondary validation diagnostic:
Yes validation dialogues: 213
Yes validation move examples: 1297
Full validation examples (PRIMARY): 1850
Filtered training rows saved: /kaggle/working/abl1_results/abl1_yes_only_train_examples.json

In [5]:
# ============================================================
# MD6 INPUT FORMAT + TOKENIZATION + ALPHA=.75 CLASS WEIGHTS
# ============================================================

def format_history(history):
    if not history:
        return "[No previous conversation]"
    return "\n".join(
        f"{turn['user']}: {turn['text']}"
        for turn in history
    )

def build_problem_text(example):
    return "Problem:\n" + str(example["problem"])

def build_conversation_text(example):
    return (
        "Conversation:\n"
        + format_history(example["history"])
        + "\n\nNext teacher pedagogical move:"
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.truncation_side = "left"

def prepare_rows(rows):
    prepared = []
    for ex in rows:
        prepared.append({
            "example_id": ex["example_id"],
            "qid": ex["qid"],
            "problem_text": build_problem_text(ex),
            "conversation_text": build_conversation_text(ex),
            "labels": LABEL2ID[ex["target_move"]],
        })
    return prepared

train_prepared = prepare_rows(train_rows)
val_prepared = prepare_rows(val_rows)

# Same fixed DistilRoBERTa length diagnostic as MD6.
val_original_lengths = []

for ex in val_rows:
    encoded = tokenizer(
        build_problem_text(ex),
        build_conversation_text(ex),
        add_special_tokens=True,
        truncation=False,
    )
    val_original_lengths.append(
        len(encoded["input_ids"])
    )

val_original_lengths = np.asarray(
    val_original_lengths
)

short_mask = val_original_lengths <= 512
long_mask = val_original_lengths > 512

print("Validation <=512:", int(short_mask.sum()))
print("Validation >512:", int(long_mask.sum()))

assert int(short_mask.sum()) == 1507
assert int(long_mask.sum()) == 343

# ------------------------------------------------------------
# Same MD6 weighting RECIPE, recalculated on ABL1 train subset.
# ------------------------------------------------------------

original_train_counts = Counter(
    r["target_move"]
    for r in train_rows_full
)

train_counts = Counter(
    r["target_move"]
    for r in train_rows
)

n = len(train_rows)
k = len(LABELS)

raw_class_weights = np.asarray(
    [
        n / (k * train_counts[label])
        for label in LABELS
    ],
    dtype=np.float32,
)

class_weights = np.power(
    raw_class_weights,
    WEIGHT_ALPHA,
).astype(np.float32)

print("\n" + "=" * 78)
print("CLASS DISTRIBUTION CHANGE")
print("=" * 78)

for label in LABELS:
    old_count = original_train_counts[label]
    new_count = train_counts[label]

    print(
        f"{label:10s} "
        f"MD6_all={old_count:4d} "
        f"ABL1_yes={new_count:4d} "
        f"retained={new_count / old_count:.2%}"
    )

print("\nABL1 alpha=.75 class weights:")

for label, raw_w, tuned_w in zip(
    LABELS,
    raw_class_weights,
    class_weights,
):
    print(
        f"{label:10s} "
        f"raw={raw_w:.6f} "
        f"tuned={tuned_w:.6f}"
    )

train_ds = Dataset.from_list(
    train_prepared
)

val_ds = Dataset.from_list(
    val_prepared
)

def tokenize_batch(batch):
    return tokenizer(
        batch["problem_text"],
        batch["conversation_text"],
        truncation="only_second",
        max_length=MAX_LENGTH,
    )

train_tok = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "problem_text",
        "conversation_text",
        "example_id",
        "qid",
    ],
)

val_tok = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "problem_text",
        "conversation_text",
        "example_id",
        "qid",
    ],
)

assert max(
    len(x)
    for x in train_tok["input_ids"]
) <= MAX_LENGTH

assert max(
    len(x)
    for x in val_tok["input_ids"]
) <= MAX_LENGTH

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("\nTokenization: PASSED")


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (578 > 512). Running this sequence through the model will result in indexing errors


Validation <=512: 1507
Validation >512: 343

CLASS DISTRIBUTION CHANGE
generic    MD6_all=3579 ABL1_yes=2749 retained=76.81%
probing    MD6_all=3413 ABL1_yes=2274 retained=66.63%
focus      MD6_all=5443 ABL1_yes=3567 retained=65.53%
telling    MD6_all=2470 ABL1_yes=1208 retained=48.91%

ABL1 alpha=.75 class weights:
generic    raw=0.891051 tuned=0.917122
probing    raw=1.077177 tuned=1.057341
focus      raw=0.686712 tuned=0.754364
telling    raw=2.027732 tuned=1.699252


Map:   0%|          | 0/9798 [00:00<?, ? examples/s]

Map:   0%|          | 0/1850 [00:00<?, ? examples/s]


Tokenization: PASSED


In [6]:
# ============================================================
# EXACT MD6 MODEL/TRAINING RECIPE
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
)

class WeightedTrainer(Trainer):
    def __init__(
        self,
        *args,
        class_weights=None,
        **kwargs,
    ):
        super().__init__(
            *args,
            **kwargs,
        )
        self.class_weights_tensor = torch.tensor(
            class_weights,
            dtype=torch.float32,
        )

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Same implementation that successfully trained MD6.
        weights = self.class_weights_tensor.to(
            logits.device
        )

        loss = F.cross_entropy(
            logits,
            labels,
            weight=weights,
        )

        if return_outputs:
            return loss, outputs

        return loss

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    pred = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(
            labels,
            pred,
        ),
        "macro_f1": f1_score(
            labels,
            pred,
            labels=list(range(len(LABELS))),
            average="macro",
            zero_division=0,
        ),
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=280,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,

    # Manual checkpoint selection is used only to avoid the old
    # Trainer auto-reload warning. It does not alter training.
    load_best_model_at_end=False,

    fp16=True,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

print("Trainer GPU count:", trainer.args.n_gpu)
assert trainer.args.n_gpu == 2

effective_global_batch = (
    8
    * trainer.args.n_gpu
    * training_args.gradient_accumulation_steps
)

steps_per_epoch = int(
    np.ceil(
        len(train_rows)
        / effective_global_batch
    )
)

print("ABL1 effective global batch:", effective_global_batch)
print("ABL1 optimizer steps/epoch:", steps_per_epoch)
print("MD6 optimizer steps/epoch reference: 932")


model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainer GPU count: 2
ABL1 effective global batch: 16
ABL1 optimizer steps/epoch: 613
MD6 optimizer steps/epoch reference: 932


In [7]:
# ============================================================
# TRAIN ABL1
# ============================================================

print("\n" + "=" * 78)
print("STARTING ABL1 TRAINING")
print("=" * 78)

trainer.train()

history_path = RESULT_DIR / "abl1_log_history.json"

with history_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        trainer.state.log_history,
        f,
        indent=2,
    )

print("\n" + "=" * 78)
print("EPOCH VALIDATION HISTORY — FULL VALIDATION")
print("=" * 78)

for row in trainer.state.log_history:
    if "eval_macro_f1" in row:
        print(
            f"Epoch {row.get('epoch')}: "
            f"accuracy={row.get('eval_accuracy'):.6f} "
            f"macro_f1={row.get('eval_macro_f1'):.6f} "
            f"val_loss={row.get('eval_loss'):.6f}"
        )

print("\nAll 3 epoch checkpoints saved.")
print(
    "Best checkpoint will be selected manually "
    "using FULL-validation Macro-F1."
)



STARTING ABL1 TRAINING


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.269097,1.259096,0.449730,0.439904
2,1.196490,1.233876,0.472973,0.446307
3,1.140499,1.222004,0.458378,0.449527


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


EPOCH VALIDATION HISTORY — FULL VALIDATION
Epoch 1.0: accuracy=0.449730 macro_f1=0.439904 val_loss=1.259096
Epoch 2.0: accuracy=0.472973 macro_f1=0.446307 val_loss=1.233876
Epoch 3.0: accuracy=0.458378 macro_f1=0.449527 val_loss=1.222004

All 3 epoch checkpoints saved.
Best checkpoint will be selected manually using FULL-validation Macro-F1.


In [8]:
# ============================================================
# EVALUATE ALL CHECKPOINTS + FULL/YES-ONLY DIAGNOSTICS
# ============================================================

def softmax_np(logits):
    z = logits - logits.max(
        axis=1,
        keepdims=True,
    )
    ez = np.exp(z)
    return ez / ez.sum(
        axis=1,
        keepdims=True,
    )

def subset_metrics(
    true_ids,
    pred_ids,
    mask,
):
    y_true = true_ids[mask]
    y_pred = pred_ids[mask]

    return {
        "n": int(mask.sum()),
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(
                    range(len(LABELS))
                ),
                average="macro",
                zero_division=0,
            )
        ),
        "per_class_f1": {
            label: float(score)
            for label, score in zip(
                LABELS,
                f1_score(
                    y_true,
                    y_pred,
                    labels=list(
                        range(len(LABELS))
                    ),
                    average=None,
                    zero_division=0,
                ),
            )
        },
    }

checkpoint_dirs = sorted(
    OUTPUT_DIR.glob("checkpoint-*"),
    key=lambda p: int(
        p.name.split("-")[-1]
    ),
)

assert len(checkpoint_dirs) == 3, (
    f"Expected 3 checkpoints, found {len(checkpoint_dirs)}"
)

all_results = []

for epoch_idx, ckpt in enumerate(
    checkpoint_dirs,
    start=1,
):
    print("\n" + "-" * 78)
    print(f"EPOCH {epoch_idx}: {ckpt}")
    print("-" * 78)

    ckpt_model = (
        AutoModelForSequenceClassification
        .from_pretrained(ckpt)
    )

    eval_args = TrainingArguments(
        output_dir=str(
            RESULT_DIR
            / f"eval_epoch{epoch_idx}"
        ),
        per_device_eval_batch_size=16,
        fp16=True,
        seed=SEED,
        data_seed=SEED,
        report_to="none",
    )

    eval_trainer = WeightedTrainer(
        model=ckpt_model,
        args=eval_args,
        eval_dataset=val_tok,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    pred_output = eval_trainer.predict(
        val_tok
    )

    logits = pred_output.predictions
    true_ids = pred_output.label_ids
    pred_ids = np.argmax(
        logits,
        axis=1,
    )
    probs = softmax_np(logits)

    overall_accuracy = float(
        accuracy_score(
            true_ids,
            pred_ids,
        )
    )

    overall_macro_f1 = float(
        f1_score(
            true_ids,
            pred_ids,
            labels=list(
                range(len(LABELS))
            ),
            average="macro",
            zero_division=0,
        )
    )

    per_class_f1 = f1_score(
        true_ids,
        pred_ids,
        labels=list(
            range(len(LABELS))
        ),
        average=None,
        zero_division=0,
    )

    report = classification_report(
        true_ids,
        pred_ids,
        labels=list(
            range(len(LABELS))
        ),
        target_names=LABELS,
        digits=6,
        zero_division=0,
        output_dict=True,
    )

    cm = confusion_matrix(
        true_ids,
        pred_ids,
        labels=list(
            range(len(LABELS))
        ),
    ).tolist()

    short_result = subset_metrics(
        true_ids,
        pred_ids,
        short_mask,
    )

    long_result = subset_metrics(
        true_ids,
        pred_ids,
        long_mask,
    )

    yes_val_result = subset_metrics(
        true_ids,
        pred_ids,
        val_yes_mask,
    )

    result = {
        "epoch": epoch_idx,
        "checkpoint": str(ckpt),

        # PRIMARY
        "full_validation": {
            "n": len(val_rows),
            "accuracy": overall_accuracy,
            "macro_f1": overall_macro_f1,
            "per_class_f1": {
                label: float(score)
                for label, score
                in zip(
                    LABELS,
                    per_class_f1,
                )
            },
            "classification_report": report,
            "confusion_matrix": cm,
        },

        # SECONDARY DIAGNOSTICS
        "le512": short_result,
        "gt512": long_result,
        "yes_only_validation": yes_val_result,
    }

    all_results.append(result)

    print("PRIMARY — FULL validation")
    print("Accuracy:", overall_accuracy)
    print("Macro-F1:", overall_macro_f1)
    print(
        "Per-class F1:",
        result["full_validation"]["per_class_f1"],
    )

    print("\nSecondary diagnostics")
    print(
        "<=512 Macro-F1:",
        short_result["macro_f1"],
    )
    print(
        ">512 Macro-F1:",
        long_result["macro_f1"],
    )
    print(
        "Yes-only validation Macro-F1:",
        yes_val_result["macro_f1"],
    )

    pred_path = (
        RESULT_DIR
        / f"abl1_epoch{epoch_idx}_predictions.jsonl"
    )

    with pred_path.open(
        "w",
        encoding="utf-8",
    ) as f:
        for i, (
            true_id,
            pred_id,
            probability_row,
        ) in enumerate(
            zip(
                true_ids,
                pred_ids,
                probs,
            )
        ):
            record = {
                "example_id": val_rows[i]["example_id"],
                "qid": val_rows[i]["qid"],
                "true_move": ID2LABEL[int(true_id)],
                "predicted_move": ID2LABEL[int(pred_id)],
                "original_distil_token_length": int(
                    val_original_lengths[i]
                ),
                "is_yes_outcome_diagnostic": bool(
                    val_yes_mask[i]
                ),
                "probabilities": {
                    label: float(
                        probability_row[j]
                    )
                    for j, label
                    in enumerate(LABELS)
                },
            }

            f.write(
                json.dumps(record)
                + "\n"
            )

    del eval_trainer
    del ckpt_model
    gc.collect()
    torch.cuda.empty_cache()

# Winner chosen ONLY on complete validation Macro-F1.
best_result = max(
    all_results,
    key=lambda x: (
        x["full_validation"]["macro_f1"]
    ),
)

summary = {
    "experiment": (
        "ABL1_MD6_self_correctness_Yes_only"
    ),

    "research_question": (
        "Does restricting MD6 training to dialogues with "
        "self-correctness='Yes' improve pedagogical move "
        "classification on the unchanged full validation set?"
    ),

    "offline_filter_only": True,
    "runtime_selector_uses_self_correctness": False,

    "model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "weight_alpha": WEIGHT_ALPHA,

    "original_train_dialogues": 2295,
    "yes_only_train_dialogues": yes_train_dialogues,
    "original_train_move_examples": len(train_rows_full),
    "yes_only_train_move_examples": len(train_rows),

    "original_train_class_counts": dict(
        original_train_counts
    ),
    "yes_only_train_class_counts": dict(
        train_counts
    ),

    "yes_only_tuned_class_weights": {
        label: float(weight)
        for label, weight
        in zip(
            LABELS,
            class_weights,
        )
    },

    "training_configuration": {
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 8,
        "effective_global_batch": 16,
        "num_train_epochs": 3,
        "weight_decay": 0.01,
        "warmup_steps": 280,
        "fp16": True,
        "seed": SEED,
        "custom_test_used": False,
    },

    "primary_evaluation": (
        "unchanged full 1850-example validation set"
    ),

    "baseline_md6_epoch2": {
        "accuracy": 0.5016216216216216,
        "macro_f1": 0.487684572655523,
        "per_class_f1": {
            "generic": 0.5875,
            "probing": 0.4179,
            "focus": 0.5710,
            "telling": 0.3744,
        },
    },

    "epochs": all_results,

    "best_epoch": best_result["epoch"],
    "best_full_validation_accuracy": (
        best_result["full_validation"]["accuracy"]
    ),
    "best_full_validation_macro_f1": (
        best_result["full_validation"]["macro_f1"]
    ),
    "best_yes_only_validation_macro_f1": (
        best_result["yes_only_validation"]["macro_f1"]
    ),
}

summary_path = (
    RESULT_DIR
    / "abl1_summary.json"
)

with summary_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
    )

best_checkpoint = Path(
    best_result["checkpoint"]
)

best_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        best_checkpoint
    )
)

best_model_dir = (
    RESULT_DIR
    / "abl1_best_model"
)

best_model.save_pretrained(
    best_model_dir
)

tokenizer.save_pretrained(
    best_model_dir
)

archive_path = shutil.make_archive(
    str(
        RESULT_DIR
        / "abl1_best_model"
    ),
    "gztar",
    root_dir=best_model_dir,
)

print("\n" + "=" * 78)
print("ABL1 COMPLETE")
print("=" * 78)

print("Best epoch:", best_result["epoch"])
print(
    "Best FULL-validation accuracy:",
    best_result["full_validation"]["accuracy"],
)
print(
    "Best FULL-validation Macro-F1:",
    best_result["full_validation"]["macro_f1"],
)
print(
    "Best YES-only validation Macro-F1:",
    best_result["yes_only_validation"]["macro_f1"],
)

print("\nDecision baseline:")
print("MD6 Epoch 2 accuracy = 0.5016216216216216")
print("MD6 Epoch 2 Macro-F1 = 0.487684572655523")

print("\nSummary:", summary_path)
print("Best model archive:", archive_path)



------------------------------------------------------------------------------
EPOCH 1: /kaggle/working/abl1_md6_yes_only/checkpoint-613
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

PRIMARY — FULL validation
Accuracy: 0.44972972972972974
Macro-F1: 0.4399037838421372
Per-class F1: {'generic': 0.5779569892473119, 'probing': 0.3722721437740693, 'focus': 0.49038461538461536, 'telling': 0.31900138696255204}

Secondary diagnostics
<=512 Macro-F1: 0.43268029703938266
>512 Macro-F1: 0.19167814825020668
Yes-only validation Macro-F1: 0.409569831008816

------------------------------------------------------------------------------
EPOCH 2: /kaggle/working/abl1_md6_yes_only/checkpoint-1226
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

PRIMARY — FULL validation
Accuracy: 0.47297297297297297
Macro-F1: 0.44630666715213413
Per-class F1: {'generic': 0.5777310924369747, 'probing': 0.34911242603550297, 'focus': 0.5290780141843971, 'telling': 0.3293051359516616}

Secondary diagnostics
<=512 Macro-F1: 0.4357714088760669
>512 Macro-F1: 0.2659009896215847
Yes-only validation Macro-F1: 0.4284584165919508

------------------------------------------------------------------------------
EPOCH 3: /kaggle/working/abl1_md6_yes_only/checkpoint-1839
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

PRIMARY — FULL validation
Accuracy: 0.45837837837837836
Macro-F1: 0.44952715415060296
Per-class F1: {'generic': 0.5909617612977984, 'probing': 0.36036036036036034, 'focus': 0.49645390070921985, 'telling': 0.35033259423503327}

Secondary diagnostics
<=512 Macro-F1: 0.4549490339849591
>512 Macro-F1: 0.22287371801512484
Yes-only validation Macro-F1: 0.42928671759989434


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


ABL1 COMPLETE
Best epoch: 3
Best FULL-validation accuracy: 0.45837837837837836
Best FULL-validation Macro-F1: 0.44952715415060296
Best YES-only validation Macro-F1: 0.42928671759989434

Decision baseline:
MD6 Epoch 2 accuracy = 0.5016216216216216
MD6 Epoch 2 Macro-F1 = 0.487684572655523

Summary: /kaggle/working/abl1_results/abl1_summary.json
Best model archive: /kaggle/working/abl1_results/abl1_best_model.tar.gz
